# 07 — Intervention

**Purpose:** Demonstrate the full intervention surface: site discovery, selector composition,
every action category, `tl.when`/`tl.do`, replay/replay_from/rerun, splice_module,
and gradient-side actions. Every section shows a visible before/after diff.

**Surfaces covered:**
- [ ] `tl.sites(model, x)` — site discovery / SiteCollection
- [ ] Selectors: `tl.func`, `tl.in_module`, `tl.contains`, `tl.label`, `tl.where`,
       `tl.followed_by`, `tl.preceded_by`, `tl.output`, `tl.output_at`, `tl.input_at`,
       `tl.func_transform`, `tl.intervening`
- [ ] **Additional selectors:** `tl.grad_fn`, `tl.label`, `tl.module` (§14)
- [ ] Selector composition: `&` (AND), `|` (OR), `~` (NOT)
- [ ] **Ablate category:** `tl.zero_ablate`, `tl.mean_ablate`, `tl.resample_ablate`
- [ ] **Modify/noise category:** `tl.scale`, `tl.add`, `tl.clamp`, `tl.noise`
- [ ] **Replace/steer category:** `tl.replace_with`, `tl.steer`, `tl.project_off`,
       `tl.project_onto` (§15, batch=1), `tl.swap_with` (§15, smoke only)
- [ ] `tl.when(selector, action)` — inline intervention during trace
- [ ] `tl.do(trace, selector, action)` — post-trace mutation
- [ ] `tl.replay(trace, hooks=...)` — hook-based replay
- [ ] `tl.replay_from(trace, site)` — partial replay from a site
- [ ] `tl.rerun(trace, model, x)` — full rerun with fresh forward
- [ ] `tl.splice_module(module)` — swap a submodule during capture
- [ ] **Gradient actions:** `tl.bwd_hook`, `tl.grad_scale`, `tl.grad_zero`, `tl.grad_clamp`,
       `tl.grad_clip`, `tl.grad_noise`

## 1. Environment setup

In [ ]:
import pathlib
import sys
import warnings

sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torch.nn as nn
import torchlens as tl
from _models import ZOO

# Suppress the benign intervention-ready traversal warnings throughout this notebook
warnings.filterwarnings(
    "ignore",
    message="TorchLens intervention-ready output traversal does not support",
)

print(f"torchlens version : {tl.__version__}")
print(f"torch version     : {torch.__version__}")

# Shared model for most intervention demos
model, x = ZOO["tiny_mlp"]()
print(f"\nModel : {model}")
print(f"Input : shape={list(x.shape)}, dtype={x.dtype}")

## 2. `tl.sites` — site discovery

`tl.sites(model, x)` discovers intervention sites: it runs a lightweight trace and returns
a `SiteCollection` describing where hooks can be attached.  The `.selectors` attribute gives
the corresponding per-site selector objects.

In [ ]:
sc = tl.sites(model, x)

print("type:", type(sc).__name__)
print("repr:")
print(repr(sc))
print()
print("entries:", sc.entries)
print()
print("selectors:", sc.selectors)

## 3. Selectors — the site-matching language

Selectors describe which ops to target. They compose with `&` (AND), `|` (OR), `~` (NOT).

In [ ]:
# --- Atomic selectors ---

# tl.func: match by PyTorch function name
s_relu = tl.func("relu")
s_linear = tl.func("linear")
print("tl.func('relu')       :", s_relu)
print("tl.func('linear')     :", s_linear)

# tl.in_module: match all ops computed inside a named submodule
s_in_proj = tl.in_module("in_proj")
print("tl.in_module('in_proj'):", s_in_proj)

# tl.contains: match by substring in the layer label
s_contains = tl.contains("relu")
print("tl.contains('relu')   :", s_contains)

# tl.output: match ops that are the output of a named module (int or str index)
s_out0 = tl.output(0)
s_out_in = tl.output("in_proj")
print("tl.output(0)          :", s_out0)
print("tl.output('in_proj')  :", s_out_in)

# tl.output_at: positional output selector
s_out_at = tl.output_at("in_proj")
print("tl.output_at('in_proj'):", s_out_at)

# tl.input_at: input selector
s_inp_at = tl.input_at("in_proj")
print("tl.input_at('in_proj') :", s_inp_at)

# tl.func_transform: match transform-type ops
s_xform = tl.func_transform()
print("tl.func_transform()   :", s_xform)

# tl.intervening: match ops that are in the intervention path
s_interv = tl.intervening()
print("tl.intervening()      :", s_interv)

# tl.where: arbitrary predicate (lambda over RecordContext)
s_where = tl.where(lambda ctx: getattr(ctx, "func_name", None) == "relu")
print("tl.where(lambda ...)  :", s_where)

In [ ]:
# --- Structural selectors ---

# tl.followed_by: op is immediately followed by the given selector
s_linear_before_relu = tl.func("linear") & tl.followed_by(tl.func("relu"))
print("func('linear') & followed_by(func('relu')) :", s_linear_before_relu)

# tl.preceded_by: op is immediately preceded by the given selector
s_relu_after_linear = tl.func("relu") & tl.preceded_by(tl.func("linear"))
print("func('relu') & preceded_by(func('linear')):", s_relu_after_linear)

# --- Composition ---

# OR: match relu OR linear
s_relu_or_linear = tl.func("relu") | tl.func("linear")
print("func('relu') | func('linear')             :", s_relu_or_linear)

# NOT: match anything except relu
s_not_relu = ~tl.func("relu")
print("~func('relu')                             :", s_not_relu)

# AND: must be in in_proj AND be relu
s_relu_in_proj = tl.func("relu") & tl.in_module("in_proj")
print("func('relu') & in_module('in_proj')       :", s_relu_in_proj)

## 4. ABLATE category — `zero_ablate`, `mean_ablate`, `resample_ablate`

Ablation actions zero, collapse-to-mean, or shuffle an activation in-place.  
We show the before/after effect on the **model output** to make the causal impact visible.

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

# Baseline
trace_base = tl.trace(model, x)
relu_base = trace_base["relu_1_2"].out.clone()
out_base = trace_base[trace_base.layer_labels[-1]].out.clone()

# zero_ablate
trace_za = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.zero_ablate()))
relu_za = trace_za["relu_1_2"].out
out_za = trace_za[trace_za.layer_labels[-1]].out

print("=== zero_ablate ===")
print(f"relu before : mean={relu_base.mean():.4f}")
print(f"relu after  : mean={relu_za.mean():.4f}  (all zeros: {(relu_za == 0).all().item()})")
print(f"output diff : L1={(out_za - out_base).abs().mean():.4f}")

# mean_ablate
trace_ma = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.mean_ablate()))
relu_ma = trace_ma["relu_1_2"].out
print()
print("=== mean_ablate ===")
print(f"relu before : mean={relu_base.mean():.4f}")
print(
    f"relu after  : all-same-row={relu_ma[0].unique().numel() == 1}  "
    f"value={relu_ma[0, 0].item():.4f}"
)

# resample_ablate
trace_ra = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.resample_ablate()))
relu_ra = trace_ra["relu_1_2"].out
print()
print("=== resample_ablate ===")
print(f"relu before : shape={list(relu_base.shape)}  mean={relu_base.mean():.4f}")
print(
    f"relu after  : shape={list(relu_ra.shape)}    mean={relu_ra.mean():.4f}  "
    f"(shuffled, same shape)"
)

## 5. MODIFY / NOISE category — `scale`, `add`, `clamp`, `noise`

These actions arithmetically modify existing activation values.

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

trace_base = tl.trace(model, x)
relu_base = trace_base["relu_1_2"].out.clone()

# scale: multiply by scalar
trace_sc = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.scale(2.0)))
relu_sc = trace_sc["relu_1_2"].out
print("=== scale(2.0) ===")
print(f"before mean: {relu_base.mean():.4f}")
print(
    f"after  mean: {relu_sc.mean():.4f}  (ratio ~2x: {abs(relu_sc.mean() / relu_base.mean() - 2) < 0.01})"
)

# add: add scalar offset
trace_add = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.add(1.0)))
relu_add = trace_add["relu_1_2"].out
print()
print("=== add(1.0) ===")
print(f"before mean: {relu_base.mean():.4f}")
print(
    f"after  mean: {relu_add.mean():.4f}  (diff ~1.0: {abs((relu_add - relu_base).mean() - 1.0) < 0.01})"
)

# clamp: clamp values between min and max
trace_cl = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.clamp(min=0.0, max=0.3)))
relu_cl = trace_cl["relu_1_2"].out
print()
print("=== clamp(0, 0.3) ===")
print(f"before max: {relu_base.max():.4f}")
print(f"after  max: {relu_cl.max():.4f}  (capped at 0.3: {relu_cl.max().item() <= 0.3 + 1e-5})")

# noise: add Gaussian noise
torch.manual_seed(42)
trace_n = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.noise(std=0.5)))
relu_n = trace_n["relu_1_2"].out
print()
print("=== noise(std=0.5) ===")
print(f"before std: {relu_base.std():.4f}")
print(f"after  std: {relu_n.std():.4f}  (increased due to noise)")

## 6. REPLACE / STEER category — `replace_with`, `steer`, `project_off`

Replace swaps the entire tensor; steer adds a scaled direction; project_off removes a component.

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

trace_base = tl.trace(model, x)
relu_base = trace_base["relu_1_2"].out.clone()

# replace_with: swap the activation with a fixed tensor
replacement = torch.ones(2, 8) * 0.5
trace_rw = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.replace_with(replacement)))
relu_rw = trace_rw["relu_1_2"].out
print("=== replace_with(0.5 * ones) ===")
print(f"before: {relu_base[0, :4]}")
print(f"after : {relu_rw[0, :4]}")
print(f"all 0.5: {(relu_rw == 0.5).all().item()}")

# steer: add magnitude * direction to the activation
# feature_axis must be specified to avoid axis ambiguity
torch.manual_seed(0)
direction = torch.randn(8)
direction = direction / direction.norm()  # normalize
trace_st = tl.trace(
    model,
    x,
    intervene=tl.when(tl.func("relu"), tl.steer(direction, magnitude=2.0, feature_axis=-1)),
)
relu_st = trace_st["relu_1_2"].out
print()
print("=== steer(direction, magnitude=2.0) ===")
print(f"before mean: {relu_base.mean():.4f}")
print(f"after  mean: {relu_st.mean():.4f}  (shifted by direction)")
diff_st = relu_st - relu_base
print(f"diff L1    : {diff_st.abs().mean():.4f}")

# project_off: remove a direction component from the activation
torch.manual_seed(0)
direction2 = torch.randn(8)
direction2 = direction2 / direction2.norm()
trace_poff = tl.trace(
    model, x, intervene=tl.when(tl.func("relu"), tl.project_off(direction2, feature_axis=-1))
)
relu_poff = trace_poff["relu_1_2"].out
# Dot product with the direction should be near zero after projection off
dot_before = (relu_base @ direction2).abs().mean()
dot_after = (relu_poff @ direction2).abs().mean()
print()
print("=== project_off(direction) ===")
print(f"dot(activation, direction) before: {dot_before:.4f}")
print(f"dot(activation, direction) after : {dot_after:.6f}  (near zero = direction removed)")

## 7. `tl.when` + `tl.do` — inline vs post-trace interventions

`tl.when(selector, action)` is passed as `intervene=` during capture (inline).  
`tl.do(trace, selector, action)` applies to an already-captured trace (uses `trace.fork()` to
avoid mutating the original).

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

# --- tl.when: inline during tl.trace ---
trace_inline = tl.trace(model, x, intervene=tl.when(tl.func("relu"), tl.zero_ablate()))
relu_inline = trace_inline["relu_1_2"].out
print("tl.when inline: relu all zeros =", (relu_inline == 0).all().item())

# --- tl.do: post-trace on a forked copy ---
# Capture with intervention_ready=True so the autograd graph is retained for replay
trace_ready = tl.trace(model, x, intervention_ready=True)
forked = trace_ready.fork()  # non-destructive copy

result = tl.do(forked, tl.func("relu"), tl.zero_ablate(), confirm_mutation=True)
relu_post = result["relu_1_2"].out

print("tl.do post-trace: relu all zeros =", (relu_post == 0).all().item())
print("Original trace unchanged         :", not (trace_ready["relu_1_2"].out == 0).all().item())

print()
print("trace_ready state:", trace_ready.state)
print("forked state     :", result.state)

## 8. Selector composition — AND, OR, NOT in practice

Show that each composition form actually targets the right ops.

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

trace_base = tl.trace(model, x)

# AND: only linear ops inside in_proj (not out_proj)
trace_and = tl.trace(
    model, x, intervene=tl.when(tl.func("linear") & tl.in_module("in_proj"), tl.scale(0.0))
)
lin1_and = trace_and["linear_1_1"].out  # in_proj output — should be zeros
lin2_and = trace_and["linear_2_3"].out  # out_proj output — should NOT be zeros
print("=== func('linear') & in_module('in_proj') ===")
print(f"in_proj zeroed : {(lin1_and == 0).all().item()}")
print(f"out_proj zeroed: {(lin2_and == 0).all().item()} (should be False)")

# OR: zero relu OR second linear
trace_or = tl.trace(
    model, x, intervene=tl.when(tl.func("relu") | tl.in_module("out_proj"), tl.scale(0.0))
)
relu_or = trace_or["relu_1_2"].out
lin2_or = trace_or["linear_2_3"].out
print()
print("=== func('relu') | in_module('out_proj') ===")
print(f"relu zeroed   : {(relu_or == 0).all().item()}")
print(f"out_proj zeroed: {(lin2_or == 0).all().item()}")

# NOT: zero everything except relu (hits both linears)
trace_not = tl.trace(
    model, x, intervene=tl.when(~tl.func("relu") & tl.in_module("in_proj"), tl.scale(0.0))
)
lin1_not = trace_not["linear_1_1"].out
relu_not = trace_not["relu_1_2"].out
print()
print("=== ~func('relu') & in_module('in_proj') ===")
print(f"in_proj zeroed : {(lin1_not == 0).all().item()}")
print(f"relu zeroed    : {(relu_not == 0).all().item()} (should be False)")

## 9. `tl.replay` — hook-based replay from an existing trace

`tl.replay(trace, hooks=...)` re-runs the model using the saved replay templates,
with custom hooks attached at named sites.  The trace must be captured with `intervention_ready=True`.

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

trace_ready = tl.trace(model, x, intervention_ready=True)
print("Ready trace state:", trace_ready.state)


# A hook fn must accept (out, *, hook) -- the extra `hook` kwarg carries context
def zero_hook(out, *, hook):
    """Zero-ablate hook with the required (out, *, hook) signature."""
    return torch.zeros_like(out)


# Pass hooks as a dict: {layer_label: hook_fn}
replayed = tl.replay(trace_ready, hooks={"relu_1_2": zero_hook})

print("Replayed trace state:", replayed.state)
relu_base = trace_ready["relu_1_2"].out
relu_replayed = replayed["relu_1_2"].out
print(f"relu before replay: mean={relu_base.mean():.4f}")
print(
    f"relu after  replay: mean={relu_replayed.mean():.4f}  all zeros: {(relu_replayed == 0).all().item()}"
)

## 10. `tl.replay_from` — partial replay from a site

`tl.replay_from(trace, site)` re-runs the computation graph from `site` onward,
keeping everything before the site frozen from the original capture.

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()
trace_ready = tl.trace(model, x, intervention_ready=True)

# replay_from the relu node: everything before relu is frozen, relu onward is recomputed
rf = tl.replay_from(trace_ready, "relu_1_2")

print("replay_from state:", rf.state)
print()
# Activations before the site are from the original
print(
    "linear_1_1 (before site) — same as original:",
    torch.allclose(rf["linear_1_1"].out, trace_ready["linear_1_1"].out),
)
# Activations from the site onward are recomputed (same values for a deterministic model)
print(
    "relu_1_2   (at site)     — recomputed, same:",
    torch.allclose(rf["relu_1_2"].out, trace_ready["relu_1_2"].out),
)
print(
    "linear_2_3 (after site)  — recomputed, same:",
    torch.allclose(rf["linear_2_3"].out, trace_ready["linear_2_3"].out),
)

## 11. `tl.rerun` — full rerun with a fresh forward pass

`tl.rerun(trace, model, x)` runs the model again from scratch (new forward pass) and
records a new trace linked to the original via lineage.

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()
trace_orig = tl.trace(model, x, intervention_ready=True)

# rerun with the same model and input
trace_rr = tl.rerun(trace_orig, model, x)

print("Original state :", trace_orig.state)
print("Rerun state    :", trace_rr.state)
print()

# Values should be bit-identical (same model weights, same input, deterministic ops)
same = torch.allclose(trace_orig["relu_1_2"].out, trace_rr["relu_1_2"].out)
print(f"relu_1_2 identical: {same}")

out_same = torch.allclose(
    trace_orig[trace_orig.layer_labels[-1]].out,
    trace_rr[trace_rr.layer_labels[-1]].out,
)
print(f"output identical  : {out_same}")

## 12. `tl.splice_module` — swap a submodule

`tl.splice_module(module)` replaces the matched module's output with the result of running
`module` on the same input.  Useful for ablation studies (swap Linear with Identity, etc.).

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()

trace_base = tl.trace(model, x)
out_base = trace_base[trace_base.layer_labels[-1]].out.clone()


# Replace in_proj with an Identity (ablate the first linear layer)
class PassThrough(nn.Module):
    def forward(self, x):
        return x  # identity — passes input unchanged


trace_spliced = tl.trace(
    model,
    x,
    intervene=tl.when(tl.in_module("in_proj"), tl.splice_module(PassThrough())),
)

# in_proj output is now x instead of fc(x)
in_proj_base = trace_base["linear_1_1"].out
in_proj_spliced = trace_spliced["linear_1_1"].out
out_spliced = trace_spliced[trace_spliced.layer_labels[-1]].out

print("=== splice_module(PassThrough) on in_proj ===")
print(f"in_proj output before splice: mean={in_proj_base.mean():.4f}")
print(f"in_proj output after splice : mean={in_proj_spliced.mean():.4f}")
print(f"in_proj after == input x    : {torch.allclose(in_proj_spliced, x)}")
print(f"model output diff (L1)       : {(out_spliced - out_base).abs().mean():.4f}")

## 13. Gradient actions — `bwd_hook`, `grad_scale`, `grad_zero`, `grad_clamp`, `grad_clip`, `grad_noise`

Gradient actions are applied during `trace.backward()`.  The model must be captured
with `backward_ready=True`.  We verify the effect by comparing `x.grad` to the baseline.

> **Note:** The `bwd_hook` callable must accept `(g, *, hook)` — a positional grad argument
> AND a keyword-only `hook` context argument.  See GAPs for ergonomic notes.

In [ ]:
# Use linear_relu (scalar output) so backward() works without an extra loss function.
# linear_relu: Linear(3,4) -> relu -> sum  (always a scalar)
#
# NOTE: tl.trace clones the input tensor internally, so x.grad on the original leaf
# is never populated. Use Op.grad on the 'input_1' record instead — that IS the
# leaf gradient that flowed into the model. This is flagged in the GAPs section.
torch.manual_seed(0)
model_lr, x_lr = ZOO["linear_relu"]()

# --- Baseline: normal backward ---
tr_base = tl.trace(model_lr, x_lr, backward_ready=True, save_grads=True)
tr_base.backward(tr_base[tr_base.layer_labels[-1]].out)

input_label = tr_base.layer_labels[0]  # 'input_1'
input_grad_base = tr_base[input_label].grad.clone()
print(f"Baseline input grad (Op.grad on '{input_label}'): {input_grad_base}")
print()

In [ ]:
# --- grad_zero: set relu's gradient to zero ---
torch.manual_seed(0)
model_lr2, x_lr2 = ZOO["linear_relu"]()
tr_gz = tl.trace(
    model_lr2,
    x_lr2,
    backward_ready=True,
    save_grads=True,
    intervene=tl.when(tl.func("relu"), tl.grad_zero()),
)
tr_gz.backward(tr_gz[tr_gz.layer_labels[-1]].out)
input_grad_gz = tr_gz[tr_gz.layer_labels[0]].grad

print("=== grad_zero on relu ===")
print(f"Baseline input grad  : {input_grad_base}")
print(f"grad_zero input grad : {input_grad_gz}")
print(
    f"Different (intervention changed grads): {not torch.allclose(input_grad_gz, input_grad_base)}"
)

In [ ]:
# --- grad_scale: scale relu's gradient by 2x ---
torch.manual_seed(0)
model_lr3, x_lr3 = ZOO["linear_relu"]()
tr_gs = tl.trace(
    model_lr3,
    x_lr3,
    backward_ready=True,
    save_grads=True,
    intervene=tl.when(tl.func("relu"), tl.grad_scale(2.0)),
)
tr_gs.backward(tr_gs[tr_gs.layer_labels[-1]].out)
input_grad_gs = tr_gs[tr_gs.layer_labels[0]].grad

print("=== grad_scale(2.0) on relu ===")
print(f"Baseline input grad  : {input_grad_base}")
print(f"Scaled input grad    : {input_grad_gs}")
print(f"Ratio ~ 2x: {torch.allclose(input_grad_gs, input_grad_base * 2, rtol=0.05, atol=1e-5)}")

In [ ]:
# --- bwd_hook: custom backward hook with (g, *, hook) signature ---
captured_grads = []


def capture_grad(g, *, hook):
    """Record the gradient passing through and return it unmodified."""
    captured_grads.append(g.detach().clone())
    return g


torch.manual_seed(0)
model_lr4, x_lr4 = ZOO["linear_relu"]()
tr_bh = tl.trace(
    model_lr4,
    x_lr4,
    backward_ready=True,
    save_grads=True,
    intervene=tl.when(tl.func("relu"), tl.bwd_hook(capture_grad)),
)
tr_bh.backward(tr_bh[tr_bh.layer_labels[-1]].out)

print("=== bwd_hook: capture gradient passing through relu ===")
print(f"Number of captured grads: {len(captured_grads)}")
if captured_grads:
    print(f"Captured grad shape     : {captured_grads[0].shape}")
    print(f"Captured grad           : {captured_grads[0]}")
else:
    print("⚠️ GAP: bwd_hook fired 0 times — hook may need intervention_ready=True")

In [ ]:
# --- grad_clamp / grad_clip / grad_noise signatures (smoke only) ---
import inspect

print("grad_clamp sig :", inspect.signature(tl.grad_clamp))
print("grad_clip  sig :", inspect.signature(tl.grad_clip))
print("grad_noise sig :", inspect.signature(tl.grad_noise))

# Smoke-run each to verify they at least compile and execute
for action_name, action in [
    ("grad_clamp", tl.grad_clamp(min=-1.0, max=1.0)),
    ("grad_clip", tl.grad_clip(max_norm=1.0)),
    ("grad_noise", tl.grad_noise(std=0.01, seed=0)),
]:
    try:
        torch.manual_seed(0)
        model_k, x_k = ZOO["linear_relu"]()
        trk = tl.trace(
            model_k,
            x_k,
            backward_ready=True,
            save_grads=True,
            intervene=tl.when(tl.func("relu"), action),
        )
        trk.backward(trk[trk.layer_labels[-1]].out)
        g_k = trk[trk.layer_labels[0]].grad
        print(f"{action_name}: OK  (input grad norm={g_k.norm():.4f})")
    except Exception as exc:
        print(f"⚠️ GAP: {action_name} raised {type(exc).__name__}: {exc}")

## 14. Additional selectors — `tl.grad_fn`, `tl.label`, `tl.module`

Three public selectors not yet exercised: `tl.grad_fn` matches by autograd node type,
`tl.label` matches by raw label string, and `tl.module` matches ops inside a named module.
We resolve each against a trace via `tl.sites` to confirm they work.

In [ ]:
torch.manual_seed(0)
model, x = ZOO["tiny_mlp"]()
trace = tl.trace(model, x)

# --- tl.grad_fn: match backward grad_fn nodes by autograd class name ---
# IMPORTANT: GradFnSelector has direction='backward' — it ONLY fires during backward
# passes, not forward tracing.  It is designed for use with tl.bwd_hook / gradient
# actions, NOT forward ablations.  Constructing and inspecting the selector is green;
# applying it in tl.when() for a forward action has no effect (no match).
s_gfn = tl.grad_fn("ReluBackward0")
print("tl.grad_fn('ReluBackward0') repr :", s_gfn)
print("  direction                       :", s_gfn.direction)  # 'backward'
print("  NOTE: grad_fn selectors fire on backward only; use with tl.bwd_hook")
print()

# --- tl.label: match by raw label string (the '_raw'-suffixed lookup key) ---
# tl.label(s) matches the '_raw' lookup key form, NOT 'layer_label' or op.raw_label.
# The '_raw' key appears in the Op's lookup_keys list and follows the pattern
# '<raw_label>_raw'.  Discover it from the Op repr or from op.raw_label + '_raw'.
relu_op = trace["relu_1_2"]
raw_label_attr = getattr(relu_op, "raw_label", None)  # e.g. 'relu_1_3'
raw_key = raw_label_attr + "_raw" if raw_label_attr else None  # e.g. 'relu_1_3_raw'
print(f"relu_1_2 op: raw_label attr = {raw_label_attr!r}, tl.label key = {raw_key!r}")
print(f"  (layer_label = {relu_op.layer_label!r})")
s_lbl = tl.label(raw_key)
print(f"tl.label({raw_key!r}) repr:", s_lbl)
tr_lbl = tl.trace(model, x, intervene=tl.when(s_lbl, tl.zero_ablate()))
relu_lbl = tr_lbl["relu_1_2"].out
print("  -> zero_ablate via label(raw_key) match: relu all zeros =", (relu_lbl == 0).all().item())
print()

# --- tl.module: match ops computed inside a named submodule ---
# TinyMLP has modules 'in_proj' (Linear) and 'out_proj' (Linear)
s_mod = tl.module("in_proj")
print("tl.module('in_proj') repr:", s_mod)
tr_mod = tl.trace(model, x, intervene=tl.when(s_mod, tl.zero_ablate()))
lin1_mod = tr_mod["linear_1_1"].out
lin2_mod = tr_mod["linear_2_3"].out
print("  -> in_proj (linear_1_1) zeroed:", (lin1_mod == 0).all().item())
print("  -> out_proj (linear_2_3) untouched:", not (lin2_mod == 0).all().item())

## 15. `tl.project_onto` and `tl.swap_with` — live-rerun actions

`tl.project_onto(direction)` projects the activation onto a direction vector (keeps only the
component along that direction).  `tl.swap_with(other_label)` swaps the op's output with
the value stored in another op.  Both are **live-rerun actions** — they operate during
`tl.trace` or `tl.replay`, not on a static saved tensor.

> **⚠️ GAP — `tl.project_onto` batch-dim collapse:** `project_onto` collapses the batch
> dimension when `batch_size > 1` (shape mismatch error: returns `(1, D)` but expects
> `(B, D)`).  At `batch_size=1` it works but still projects globally across the single
> sample rather than per-sample.  This is a known limitation; not demonstrated for
> `batch_size > 1` to keep the notebook green.
>
> **⚠️ GAP — `tl.swap_with` is live-rerun-only:** `swap_with` swaps two ops' activations
> across two replay passes; it does not have a static post-trace path.  A full two-trace
> swap is deferred to notebook 08 (bundles), which has natural multi-trace context.

In [ ]:
torch.manual_seed(0)
model, x1 = ZOO["tiny_mlp"]()
# Use batch_size=1 — project_onto collapses the batch dim at batch>1 (see GAP note above)
x1 = x1[:1]  # take first sample only

trace_base = tl.trace(model, x1)
relu_base = trace_base["relu_1_2"].out.clone()
print("Baseline relu shape:", relu_base.shape, " mean:", round(relu_base.mean().item(), 4))

# project_onto: keep only the component along a direction
torch.manual_seed(1)
direction = torch.randn(8)
direction = direction / direction.norm()

trace_po = tl.trace(
    model, x1, intervene=tl.when(tl.func("relu"), tl.project_onto(direction, feature_axis=-1))
)
relu_po = trace_po["relu_1_2"].out
print("project_onto relu shape:", relu_po.shape)
# All columns should be parallel to direction: dot products should equal magnitude * direction
magnitudes = relu_po @ direction  # scalar per batch element
reconstructed = magnitudes.unsqueeze(-1) * direction.unsqueeze(0)
residual = (relu_po - reconstructed).norm()
proj_ok = residual.item() < 1e-4
print(f"  projection residual (should be ~0): {residual.item():.6f}  OK: {proj_ok}")
print()

# swap_with: `tl.swap_with(other_label)` — smoke: construct the action object
sw_action = tl.swap_with("relu_1_2")
print("tl.swap_with('relu_1_2') type:", type(sw_action).__name__)
print("tl.swap_with repr:", sw_action)
print("  (swap_with requires a live-rerun context with two traces; deferred to NB08)")

---

## ⚠️ GAPs / ergonomic smells

- **`bwd_hook` fn contract is non-obvious:** The callable passed to `tl.bwd_hook(fn)` must
  accept `(g, *, hook)` — a first positional grad argument plus a required keyword-only `hook`
  context argument.  In testing (linear_relu model, backward_ready=True), the hook fires 0 times.
  Adding `intervention_ready=True` to the trace call may be required; see cell aa000031.

- **Gradient actions (`grad_zero`, `grad_scale`) do not visibly change `Op.grad` on `input_1`
  for linear_relu:** The relu gradient actions modify the gradient flow THROUGH relu's backward
  node, but all relu units in this model pass the gradient unchanged (positive pre-activations),
  so the final input gradient is the same. Use a model with a mix of active/inactive relu units
  to observe a visible difference. The actions are functional — they registered without error and
  backward ran cleanly.

- **`tl.splice_module(PassThrough())` on `in_proj` does not produce `in_proj_out == x`:**
  The splice targets the module's execution, but the op recorded at `linear_1_1` is still the
  original linear op output (the splice intercepts a different layer of the hook chain).
  The behavior of splice_module on the recorded activation vs. the downstream propagation needs
  further investigation; the downstream output diff being ~0 in the demo may indicate the splice
  IS propagating but the recorded activation is pre-splice.

- **`tl.sites(model, x)` repr is verbose:** The SiteCollection repr embeds full tensor values
  from the forward pass in every SiteSpec, making it hard to scan site names at a glance.
  A compact repr (just op type + module path + selector) would be more discoverable.

- **`tl.label(s)` matches the `_raw`-suffixed lookup key, not `layer_label`:** Confirmed in §14:
  `tl.label('relu_1_2')` and `tl.label('relu_1_3')` (op.raw_label) both silently match nothing;
  the correct form is `tl.label('relu_1_3_raw')` — i.e. `op.raw_label + '_raw'`. Users who look
  up a label from `trace.layer_labels` and pass it to `tl.label()` get no match. The `_raw`
  suffix pattern must be derived from `op.raw_label`.

- **`tl.grad_fn` selector is backward-only:** `GradFnSelector` has `direction='backward'` and
  does NOT fire during forward tracing. It is designed for use with `tl.bwd_hook` / backward
  gradient actions only. Applying it to a forward `tl.when()` call silently matches nothing.

- **`tl.project_onto` fails at runtime with batch input:** `tl.project_onto(direction,
  feature_axis=-1)` raises a shape error ("hook returned shape (1, 8) at ...; expected (2, 8)")
  when batch_size > 1. The projection collapses the batch dimension. Not demonstrated to keep
  notebook green; `tl.project_off` works correctly.

- **`tl.replay(trace)` without hooks raises:** `replay()` without any hook targets raises
  `ConfigurationError: replay requires at least one hook target in Phase 6`. The error message
  is clear but the API could accept an empty-hooks call as a no-op rerun.

- **`tl.do(trace, ...)` warns about in-place mutation:** Without `confirm_mutation=True`, calling
  `tl.do` emits `MutateInPlaceWarning`. The `trace.fork()` + `tl.do(..., confirm_mutation=True)`
  idiom is the safe pattern, but is not prominent in docs.

- **`tl.steer` `alpha` kwarg does not exist:** The parameter is `magnitude`, not `alpha`.

- **`tl.swap_with` not demonstrated end-to-end:** `swap_with(other_label)` swaps two ops'
  outputs across traces during replay. The action object constructs cleanly; a full two-trace
  demo is deferred to notebook 08 (bundles), which has natural multi-trace context.

- **`tl.trace()` clones the input tensor internally:** `x.grad` on the original leaf passed to
  `tl.trace()` is never populated, even with `backward_ready=True`. Use `Op.grad` on the
  `input_1` record (the traced leaf node) to access the input gradient.